### Unificación de datos

El siguiente notebook se encarga de mezclar todos los archivos csv de datos de buses y metro, para crear dos nuevos archivos csv con estos datos unificados.

In [7]:
import pandas as pd
import pandas.errors
from os import listdir
from os.path import join

buses_csvs = listdir(join("..", "buses_outputs"))
metro_csvs = listdir(join("..", "metro_outputs"))

if ".ipynb_checkpoints" in buses_csvs:
    buses_csvs.remove(".ipynb_checkpoints")
if ".ipynb_checkpoints" in metro_csvs:
    metro_csvs.remove(".ipynb_checkpoints")

unified_bus_df_list = []
unified_metro_df_list = []

for bus_csv in buses_csvs:
    bus_csv_df = pd.read_csv(join("..", "buses_outputs", bus_csv))
    unified_bus_df_list.append(bus_csv_df)

for metro_csv in metro_csvs: # Habemus un problemón los datos desde el 25-10-25 están bugueados
    try:
        metro_csv_df = pd.read_csv(join("..", "metro_outputs", metro_csv))
    except pandas.errors.EmptyDataError:
        continue
    unified_metro_df_list.append(metro_csv_df)

unified_bus_df = pd.concat(unified_bus_df_list, ignore_index=True, sort=False)
unified_metro_df = pd.concat(unified_metro_df_list, ignore_index=True, sort=False)
unified_bus_df.drop(columns=["X", "Y"], inplace=True)

Añadiremos datos de cada paradero al dataframe.

In [8]:
import json
# Abrimos el archivo donde tenemos información importantes de los pasajeros
ruta_json = join("..", "data", "Paraderos-Santiago-Chile.geojsonl.json")
# Guardamos su informacion en una lista
datos = []
with open(ruta_json, "r", encoding="utf-8-sig") as f:
    for archivo in f:
        datos.append(json.loads(archivo.strip()))
# creamos un datafraem con los datos de la lista
df_paradero = pd.json_normalize(datos)
# Renombramos columnas que estarán en nuestro DataFrame
df_paradero = df_paradero.rename(columns={"properties.ID": "id", 
                            "properties.CODINFRA": "codinfra",
                            "properties.SIMT": "bus_stop_code",
                            "properties.COMUNA": "comuna",
                            "properties.NOMBRE_PAR": "nombre_par",
                            "properties.NSERVICIOS": "n_servicios",
                            "properties.SERVICIOS": "servicios",
                            "geometry.coordinates": "coordinates"})
# Seleccionamos dichas columnas
df_paradero = df_paradero[["id", "codinfra", "comuna", "bus_stop_code", "nombre_par", "n_servicios", "servicios", "coordinates"]]
# Reemplazamos nombres de comunas pues el archivo venía con un encoding incorrecto
df_paradero["comuna"] = df_paradero["comuna"].replace(
    {"CONCHAL�": "CONCHALÍ",
    "ESTACI�N CENTRAL": "ESTACIÓN CENTRAL",
    "MAIP�": "MAIPÚ",
    "PE�ALOL�N": "PEÑALOLÉN",
    "SAN JOAQU�N": "SAN JOAQUÍN",
    "SAN RAM�N": "SAN RAMÓN",
    "�U�OA": "ÑUÑOA"
    })
df_paradero.head()
df_paradero_copy = df_paradero.copy()
df_paradero_copy["servicios"] = df_paradero_copy["servicios"].str.replace(r"([IR])(?=;|$)", "", regex=True)
df_paradero_copy.to_csv(join("..", "data", "paraderos.csv"), index=False)

Reparamos las coordenadas de cada paradero.

In [9]:
unified_bus_df = pd.merge(unified_bus_df, df_paradero, on= "bus_stop_code", how="inner")
unified_bus_df[["lan", "lon"]] = pd.DataFrame(unified_bus_df["coordinates"].tolist(), index=unified_bus_df.index)
unified_bus_df.drop(columns=["coordinates"], inplace= True)

Podemos notar al revisar este DataFrame, en la columna 'servicios', que todos los códigos de micro tienen una letra al final (Sabemos que puede ser N, C, E, V) pero también las hay en casos que no corresponden. En consecuencia, es necesario que los datos correspondientes a los recorridos sean los correctos-

In [10]:
clean_unified_bus_df = unified_bus_df.copy()
clean_unified_bus_df["servicios"] = clean_unified_bus_df["servicios"].str.replace(r"([IR])(?=;|$)", "", regex=True)
print(clean_unified_bus_df.head())
clean_unified_bus_df["date"] = pd.to_datetime(clean_unified_bus_df["date"])
clean_unified_bus_df["hour_minute"] = clean_unified_bus_df["date"].dt.time
clean_unified_bus_df["date_only"] = clean_unified_bus_df["date"].dt.strftime("%Y-%m-%d")
clean_unified_bus_df = clean_unified_bus_df.drop(columns=["date"])
#print(clean_unified_bus_df.head())

  bus_stop_code route_id   bus_id  meters_distance  min_arrival_time  \
0        PB1013      B12  SKDG-90             7520                16   
1        PB1013      B33  SKPP-55             2951                 8   
2         PI725      I10  FLXK-93             2315                 7   
3         PI725      I10  FLXJ-25             6384                18   
4         PI725      I02  FLXY-79             4575                12   

   max_arrival_time                        date    id      codinfra  \
0                24  2025-10-12 19:01:28.530087  8983  L-6-39-30-OP   
1                10  2025-10-12 19:01:28.530094  8983  L-6-39-30-OP   
2                 9  2025-10-12 19:01:30.454781  4883  L-13-68-5-OP   
3                26  2025-10-12 19:01:30.454787  4883  L-13-68-5-OP   
4                16  2025-10-12 19:01:30.454789  4883  L-13-68-5-OP   

      comuna                           nombre_par  n_servicios     servicios  \
0  QUILICURA         San Ignacio / esq. Galvarino           

Por último, exportamos los dataframes en csv a la carpeta csv_unificado_{servicio}.

In [11]:
clean_unified_bus_df.to_csv(join("..", "csv_unificado_buses", "unified_bus_data.csv"), index=False)
unified_metro_df.to_csv(join("..", "csv_unificado_metro", "unified_metro_data.csv"), index=False)